In [16]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [17]:
words = open("files/names.txt",'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [18]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
vocab_size

27

In [19]:
import random 
random.seed(42)
random.shuffle(words)

In [20]:
block_size = 3 # context lenght how many chars we take to predict the next one

def build_dataset(words):
    X,Y = [],[]

    for w in words:
        context = [0] * block_size

        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])


torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [29]:
class Linear:
    def __init__(self,fan_in,fan_out,bias = True):
        self.weights = torch.randn((fan_in,fan_out),generator=g) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out,) if bias else None

    def __call__(self,x):
        self.out = x @ self.weights
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self,dim,eps=1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # parameters traine with backprop
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        # **buffers** -> running mean and variance for inference (trained with momentum update (janky))
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):
        # calculate forward pass
        if self.training:
            xmean = x.mean(0,keepdim=True)
            xvar = x.var(0,keepdim=True,unbiased=True)

        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize unit variance 
        self.out = self.gamma * xhat + self.beta # scale and shift

        if self.training:
            with torch.no_grad():
                # Update the bufferes using Exponential Moving Average (EMA) with momentum
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out

    def parameters(self):
        return [self.gamma,self.beta]


class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []

class Embedding:
    def __init__(self,num_embeddings,embeddings_dim):
        self.weight = torch.randn((num_embeddings,embeddings_dim))

    def __call__(self,IX):
        self.out = self.weight[IX]        
        return self.out

    def parameters(self):
        return [self.weight]

class Flatten:
    def __call__(self,x):
        self.out = x.view(x.shape[0],-1)
        return self.out

    def parameters(self):
        return []

class Sequential:
    def __init__(self,layers):
        self.layers = layers

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]        

In [22]:
torch.manual_seed(42)

In [30]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

# C = torch.randn((vocab_size,n_embd),generator=g)
model = Sequential([
    Embedding(vocab_size,n_embd),
    Flatten(),
    Linear(n_embd * block_size,n_hidden),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,vocab_size),
])

with torch.no_grad():
    layers[-1].weights *= 0.1 # last layer make less confident

# parameters =  [p for layer in layers for p in layer.parameters()]
parameters = model.parameters()
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True      

12297


In [28]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    # mini batch construct
    ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator=g)
    Xb,Yb = Xtr[ix],Ytr[ix] #batch X,Y

    # emb = C[Xb] 
    # x = emb.view(emb.shape[0],-1)

    # forward pass
    x = Xb
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,Yb)

    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update
    lr = 0.1 if i < 100000 else 0.01 # step learing rate decay
    for p in parameters:
        p.data +=  -lr*p.grad

    if i % 10000 == 0: 
        print(f"{i:7d}/{max_steps:7d}: {loss.item():.4f}")
    lossi.append(loss.log10().item())
    break

      0/ 200000: 3.2911


In [ ]:
for layer in layers: # set training as false to evaluate the trained neural networks for batchnorm layers
    layer.training = False

In [ ]:
torch.no_grad()
def split_loss(split):
    X,y = {
        "train":(Xtr,Ytr),
        "val":(Xdev,Ydev),
        "test":(Xte,Yte),
    }[split]

    emb = C[X]
    embcat = emb.view(emb.shape[0],-1)
    for layer in layers:
        x = layer(embcat)
    
    loss = F.cross_entropy(x,y)
    print(split,loss.item())

split_loss('train')
split_loss('val')

In [ ]:
for _ in range(20):
    out = []
    context = [0] * block_size # initalize with all . . .
    while True:
        # Forward Pass
        emb = C[torch.tensor(context)] # (1,block_size,n_embd)
        embcat = emb.view(emb.shape[0],-1)
        for layer in layers:
            x = layer(embcat)
        logits = x
        probs = F.softmax(logits,1)
        # Sample from the distribution
        ix = torch.multinomial(probs,num_samples=1,generator=g).item()
        # update the context 
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:break

    print(''.join(itos[i] for i in out))
        
    